In [ ]:
#If not installed
#!pip install transformers torch

In [ ]:
#If not installed
#!pip install -U langchain-huggingface

In [1]:
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate



In [2]:
# Load the model and tokenizer locally
model_name = "google/flan-t5-base"  # You can also use "google/flan-t5-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Create a text generation pipeline
pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cpu


In [3]:
#To control generation settings
generation_kwargs = {
    "temperature": 0.7,   # Controls randomness (higher = more creative)
    #"max_length": 512,    # Max number of tokens in response
    "max_new_tokens": 20, # max tokens
    #"min_length": 150,      # Forces at least 150 words (~150 tokens)
    "top_p": 0.9,         # Nucleus sampling (higher = more diverse responses)
    "top_k": 50,          # Limits the number of top tokens considered
    "repetition_penalty": 1.2,  # Penalizes repetition (1.0 = no penalty)
    "do_sample": True,    # Enables sampling (for creative responses)
}


In [4]:
# Wrap pipeline in LangChain's HuggingFacePipeline with parameters
llm = HuggingFacePipeline(pipeline=pipe, model_kwargs=generation_kwargs)

In [5]:
# Define prompt template
template = """Question: {question}
Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=["question"])

In [6]:
# Example questions
questions = [
    "Explain the concept of black holes in simple terms.",
    "What are the main causes of climate change, and how can we address them?",
    "Provide a brief overview of the history of artificial intelligence."
]

In [ ]:
#RunnableSequence
# from langchain.schema.runnable import RunnableSequence
chain = prompt | llm

for q in questions:
     print(f"\nQ: {q}")
     print(chain.invoke({"question": q}))

In [ ]:
##Creating separate llmchain instances - using different models

In [8]:
# Define prompt
template = "Question: {question}\nAnswer: Let's think step by step."
prompt = PromptTemplate(template=template, input_variables=["question"])

In [9]:
# Load the model and tokenizer locally
model1_name = "google/flan-t5-base"  # You can also use "google/flan-t5-xl"
#model2_name = "tiiuae/falcon-7b-instruct"
model2_name = "google/flan-t5-large"


In [10]:
# Create a text generation pipeline
pipe1 = pipeline(
    "text2text-generation",
    model=model1_name,
    torch_dtype=torch.float32,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

Device set to use cpu


In [ ]:
#If using falcon, we could use this, note the change in task
"""
pipe2 = pipeline(
    "text-generation",
    model=model2_name,
    torch_dtype=torch.float32,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)
"""

In [12]:
pipe2 = pipeline(
    "text2text-generation",
    model=model2_name,
    torch_dtype=torch.float32,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

Device set to use cpu


In [13]:
llm1 = HuggingFacePipeline(pipeline=pipe1)
llm2= HuggingFacePipeline(pipeline=pipe2)

In [14]:
chain1 = prompt | llm1
chain2 = prompt | llm2



In [15]:
# Use a model based on user choice
question = "What is quantum mechanics?"
model_choice = "flan-t5"  # Example of selecting a model dynamically or asing user to provide an input

In [ ]:
if model_choice == "flan-t5":
    response = chain1.invoke({"question": question})
else:
    response = chain2.invoke({"question": question})

print(response)

In [17]:
#Using a Function to Dynamically Select the Model (uncmment below code)
#Shared Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")  # The input variable is "input"
])

#Function to create a HuggingFacePipeline for any model
def create_hf_llm(model_name: str):
    pipe = pipeline(
        "text2text-generation",
        model=model_name,
        torch_dtype=torch.float32,
        device=0 if torch.cuda.is_available() else -1
    )
    return HuggingFacePipeline(pipeline=pipe)

#Instantiate models
# Map model names to HuggingFace model IDs
model_map = {
    "flan-t5-base": "google/flan-t5-base",
    "flan-t5-large": "google/flan-t5-large"
}

# Wrap them as HuggingFacePipeline objects
llm_map = {name: create_hf_llm(path) for name, path in model_map.items()}

# Build LCEL-style chains: prompt | llm
chain_map = {name: prompt | llm for name, llm in llm_map.items()}


Device set to use cpu
Device set to use cpu


In [18]:
#Function to dynamicaly invoke the chain
def ask_question(question: str, model_choice: str):
    if model_choice not in chain_map:
        raise ValueError(f"Model {model_choice} not found. Choose from {list(chain_map.keys())}")

    chain = chain_map[model_choice]
    response = chain.invoke({"input": question})  # key must match prompt variable
    return response

In [19]:
question = "Explain relativity in simple terms."

response_base = ask_question(question, "flan-t5-base")
print("FLAN-T5-Base:", response_base)

response_large = ask_question(question, "flan-t5-large")
print("FLAN-T5-Large:", response_large)

FLAN-T5-Base: A compass is used to measure the distance between two points on a compass.
FLAN-T5-Large: Relativity is the law that states that the mass of an object is proportional to the distance between the object and the observer.


#### HuggingFacePipeline → local models
#### AzureChatOpenAI → Azure-hosted GPT models (gpt-4.1 now, gpt-5.1 later)

In [21]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os

In [22]:
#Shared Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")
])

In [27]:
from dotenv import load_dotenv
load_dotenv()

def create_azure_llm(deployment_name: str):
    return AzureChatOpenAI(
        api_key=os.getenv("API_KEY"),
        api_version=os.getenv("AZURE_API_VERSION"),
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        deployment_name=deployment_name,
        temperature=0.1,
        max_tokens=512
    )

In [28]:
#Model mapping
model_map = {
    "gpt-4.1-small": "gpt-4.1",   # deployment name in Azure
    "gpt-4.1-large": "gpt-4.1"    # you can point both to same for now
}

In [25]:
#later we can switch to use a different model and update map
#"gpt-5.1": "gpt-5.1"

In [29]:
llm_map = {name: create_azure_llm(dep) for name, dep in model_map.items()}

In [30]:
#Create chains
chain_map = {name: prompt | llm for name, llm in llm_map.items()}

In [31]:
#Dynamic invocation function
def ask_question(question: str, model_choice: str):
    if model_choice not in chain_map:
        raise ValueError(f"Model {model_choice} not found. Choose from {list(chain_map.keys())}")

    chain = chain_map[model_choice]
    response = chain.invoke({"input": question})

    return response.content  # IMPORTANT difference

In [ ]:
#Usage
question = "Explain relativity in simple terms."

response_1 = ask_question(question, "gpt-4.1-small")
print("Model 1:", response_1)

response_2 = ask_question(question, "gpt-4.1-large")
print("Model 2:", response_2)

In [33]:
#To switch later 
model_map = {
    "fast": "gpt-4.1",
    "smart": "gpt-5.1"
}

In [34]:
#We can have improved version
from dotenv import load_dotenv
load_dotenv()

def create_azure_llm(deployment_name: str, temp: float):
    return AzureChatOpenAI(
        api_key=os.getenv("API_KEY"),
        api_version=os.getenv("AZURE_API_VERSION"),
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        deployment_name=deployment_name,
        temperature=temp,
        max_tokens=512
    )

llm_map = {
    "precise": create_azure_llm("gpt-4.1", 0.1),
    "creative": create_azure_llm("gpt-4.1", 0.7)
}

### Build a multi-step reasoning pipeline

* Instead of Question → Answer
##### Question 
*   → Step 1: Understand / break down
*  → Step 2: Reason / solve
*   → Step 3: Final answer

In [36]:
#Create a pipeline of steps using LangChain Expression Language (LCEL).

In [38]:
from langchain_openai import AzureChatOpenAI
import os, dotenv
from dotenv import load_dotenv
load_dotenv()

llm = AzureChatOpenAI(
    api_key=os.getenv("API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    deployment_name=os.getenv("AZURE_DEPLOYMENT_NAME"),
    temperature=0.2,
    max_tokens=700
)

In [39]:
from langchain_core.prompts import ChatPromptTemplate

# Step 1: Break down problem
decompose_prompt = ChatPromptTemplate.from_messages([
    ("system", "Break the question into smaller logical steps."),
    ("human", "{input}")
])

# Step 2: Solve step-by-step
reason_prompt = ChatPromptTemplate.from_messages([
    ("system", "Solve the problem step by step using the plan."),
    ("human", "Question: {input}\nPlan: {plan}")
])

# Step 3: Final clean answer
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Provide a clear and concise final answer."),
    ("human", "Question: {input}\nSolution: {solution}")
])

In [40]:
#Building multi-step chain
from langchain_core.runnables import RunnablePassthrough

# Step 1: create plan
plan_chain = decompose_prompt | llm

# Step 2: reasoning
reason_chain = reason_prompt | llm

# Step 3: final answer
final_chain = final_prompt | llm

In [46]:
#Combining all steps
"""
Conceptually:

multi_step_chain = (
    wrap_input
    | plan_step
    | prepare_for_reasoning
    | reasoning_step
    | prepare_for_final
    | final_step
)
"""
from langchain_core.runnables import RunnableLambda
'''
RunnableLambda: Used to > transform data, reshape inputs/outputs
'''
multi_step_chain = (
    # Step 0: Wrap input
    RunnableLambda(lambda x: {"input": x})

    
    # Step 1: Generate plan
    '''
    This is a RunnableParallel, It runs multiple things in parallel and returns a dict
    '''
    | {
        "input": lambda x: x["input"],
        "plan": plan_chain
    }

    # Step 2: Prepare inputs for reasoning
    '''
    Converts AIMessage → string
    '''
    | RunnableLambda(lambda x: {
        "input": x["input"],
        "plan": x["plan"].content   # extract text from AIMessage
    })

    # Step 3: Run reasoning
    '''
    Again Parallel execution
    '''
    | {
        "input": lambda x: x["input"],
        "plan": lambda x: x["plan"],
        "solution": reason_chain
    }

    # Step 4: Prepare inputs for final answer
    | RunnableLambda(lambda x: {
        "input": x["input"],
        "solution": x["solution"].content
    })

    # Step 5: Final answer
    | final_chain
)

In [ ]:
response = multi_step_chain.invoke("Explain black holes in simple terms")
print(response.content)

In [48]:
#More controllable Approach
def multi_step_reasoning(question: str):
    # Step 1: Plan
    plan = (decompose_prompt | llm).invoke({"input": question}).content

    # Step 2: Reason
    solution = (reason_prompt | llm).invoke({
        "input": question,
        "plan": plan
    }).content

    # Step 3: Final Answer
    final = (final_prompt | llm).invoke({
        "input": question,
        "solution": solution
    }).content

    return {
        "plan": plan,
        "solution": solution,
        "final_answer": final
    }

In [ ]:
result = multi_step_reasoning("Why do planets orbit the sun?")
print(result["final_answer"])

In [50]:
#Using multiple llms and map
def create_llm(deployment_name):
    return AzureChatOpenAI(
        api_key=os.getenv("API_KEY"),
        api_version=os.getenv("AZURE_API_VERSION"),
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        deployment_name=deployment_name,
        temperature=0.2,
        max_tokens=700
    )

llm_map = {
    "fast": create_llm("gpt-4.1"),
    "smart": create_llm("gpt-4.1")  # future
}

In [53]:
def multi_step_reasoning(question: str, model_choice="fast"):
    llm = llm_map[model_choice]

    plan = (decompose_prompt | llm).invoke({"input": question}).content

    solution = (reason_prompt | llm).invoke({
        "input": question,
        "plan": plan
    }).content

    final = (final_prompt | llm).invoke({
        "input": question,
        "solution": solution
    }).content

    return {
        "plan": plan,
        "solution": solution,
        "final_answer": final
    }

In [54]:
result = multi_step_reasoning("Do planets orbit the sun?")
print(result["final_answer"])

Yes, planets orbit the sun due to its gravitational force.
